## Simple Calculator
Here we can use the Python SDK to develop the simple calculator agent, then save the agent to a config.yaml and run it from there.

In [1]:
import os
import sys

# Import the NeMo-Agent-Toolkit module
module_path = os.path.abspath('../../../src/')
if module_path not in sys.path:
    sys.path.insert(0, module_path)

In [ ]:
from nat.agent.sdk import NatReActAgent
from nat.llm.sdk import NimLLM
from nat.tool.sdk import CurrentTimeTool
from nat.utils.sdk.nat_workflow import NatWorkflow
from nat_simple_calculator.sdk import CalculatorToolGroup

llm = NimLLM(
    model_name="nvdev/meta/llama-3.1-70b-instruct",
    temperature=0.0,
    max_tokens=1024,
    name="nim_llm",
)

current_time_tool = CurrentTimeTool(
    name="current_datetime",
)
calculator_tool_group = CalculatorToolGroup(
    name="calculator",
)

agent = NatReActAgent(
    tools=[current_time_tool, calculator_tool_group],
    llm=llm,
    verbose=True,
    parse_agent_response_max_retries=3,
)

nat_workflow = NatWorkflow(
    entrypoint=agent,
)

/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
await nat_workflow.prompt('What is 4 * 50 plus the current hour?')

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


'215.0'

In [ ]:
import os
from pathlib import Path

path_to_yaml = Path(os.getcwd(), "config", "config.yaml").resolve()

# Create the config directory if it doesn't exist
if not path_to_yaml.parent.exists():
    os.makedirs(path_to_yaml.parent)

# Save the workflow to a config file
nat_workflow.save_to_config_file(path_to_yaml)

# Print out the config file content
with open(path_to_yaml) as f:
    print(f.read())

functions:
  current_datetime_ac57cfa0ffb340e98770e995045cdc8d:
    _type: current_datetime

function_groups:
  calculator_894f228c75b54f71b08249d55a3ef32d:
    _type: calculator

llms:
  nim_llm:
    _type: nim
    model: nvdev/meta/llama-3.1-70b-instruct
    max_tokens: 1024
    temperature: 0.0

workflow:
  _type: react_agent
  llm_name: nim_llm
  verbose: true
  tool_names:
  - current_datetime_ac57cfa0ffb340e98770e995045cdc8d
  - calculator_894f228c75b54f71b08249d55a3ef32d
  parse_agent_response_max_retries: 3



In [ ]:
from pathlib import Path

from nat.eval.sdk import RagasEvaluator
from nat.utils.sdk.nat_evaluation import EvalDatasetJsonConfig
from nat.utils.sdk.nat_evaluation import NatEvaluation

path_to_dataset = Path(
    os.path.curdir,
    "../../../../",
    "examples/getting_started/simple_calculator/src/nat_simple_calculator/data",
    "simple_calculator.json",
).resolve()

accuracy_evaluator = RagasEvaluator(
    llm=llm,
    metric="AnswerAccuracy",
    name="accuracy"
)

evaluation = NatEvaluation(
    output_dir=Path(".tmp/nat/examples/simple_calculator/"),
    dataset=EvalDatasetJsonConfig(file_path=path_to_dataset),
    evaluators=[accuracy_evaluator],
)

nat_workflow.add_evaluator(evaluation)


In [ ]:
path_to_yaml = Path(os.getcwd(), "config", "eval_config.yaml").resolve()

# Create the config directory if it doesn't exist
if not path_to_yaml.parent.exists():
    os.makedirs(path_to_yaml.parent)

# Save the workflow to a config file
nat_workflow.save_to_config_file(path_to_yaml)

# Print out the config file content
with open(path_to_yaml) as f:
    print(f.read())


In [ ]:
await nat_workflow.evaluate()
